# Ghost region pruning: translucent Gaussians in black-background regions

Open this notebook in VS Code and connect it to a **Google Colab GPU runtime**. Run the cells from top to bottom. Colab clones or fast-forwards the repository, so **commit and push the code you want to test to the repository's default branch first**.

The experiment targets translucent ghost copies: faint, slightly shifted duplicates that are most visible where the reference image is black. It works on regions rather than single Gaussians:

1. **Eligibility safeguard:** only Gaussians with opacity ≤ `TRANSLUCENT_MAX_OPACITY` (0.4) can be removed.
2. **Flagged pixels:** pixels where the reference is nearly black but the render is clearly brighter.
3. **Evidence:** translucent Gaussians whose centers project into flagged pixels on the selection views.
4. **Clusters:** evidence-seeded 26-connected voxel clusters of translucent Gaussians.
5. **Ablation:** each cluster is removed alone and kept only if flagged pixels drop on the selection views without an MAE regression.
6. **Held-out check:** accepted clusters are removed together and compared with an opacity-matched baseline on views never used for selection.

Outputs go to `report/images/ghost_region_pruning/`. Flagged pixels are marked in red, and accepted cluster centers in cyan.

In [ ]:
import os
import pathlib
import subprocess

repo_url = 'https://github.com/jon-of-us/gaussian-splatting-homwork.git'
assignment_root = pathlib.Path('/content/hw1_2026_student')

if (assignment_root / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(assignment_root), 'pull', '--ff-only'],
        check=True,
    )
elif assignment_root.exists():
    raise FileExistsError(
        f'{assignment_root} exists but is not a Git checkout. '
        'Rename it in the Colab runtime, then rerun this cell.'
    )
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', repo_url, str(assignment_root)],
        check=True,
    )

required_files = [
    assignment_root / 'gaussian_model.py',
    assignment_root / 'submodules/diff-gaussian-rasterization/cuda_rasterizer/forward.cu',
    assignment_root / 'scripts/setup_environment.sh',
    assignment_root / 'scripts/ghost_region_pruning.py',
    assignment_root / 'assets/gs_cloud.ply',
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Repository checkout is missing required files: {missing_files}')

os.chdir(assignment_root)
print(f'Assignment directory: {pathlib.Path.cwd()}')

In [ ]:
import pathlib
import subprocess
import sys
import torch
import torchvision
import torch.utils.cpp_extension

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Connect to a Colab GPU runtime, then rerun.')
if torch.utils.cpp_extension.CUDA_HOME is None:
    raise RuntimeError('CUDA Toolkit was not found. Use a Colab GPU runtime with nvcc installed.')

subprocess.run(['nvidia-smi'], check=True)
nvcc = pathlib.Path(torch.utils.cpp_extension.CUDA_HOME) / 'bin' / 'nvcc'
if not nvcc.is_file():
    raise FileNotFoundError(f'CUDA compiler was not found: {nvcc}')
subprocess.run([str(nvcc), '--version'], check=True)

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'PyTorch CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Build the CUDA extension

This uses the repository setup helper and the notebook kernel's Python. Run it once per fresh Colab runtime.

In [ ]:
import os
import sys
import subprocess

build_env = os.environ.copy()
build_env['HW1_PYTHON'] = sys.executable
build_env['MAX_JOBS'] = '2'
subprocess.run(
    ['bash', 'scripts/setup_environment.sh'],
    cwd=assignment_root,
    env=build_env,
    check=True,
)

## Configuration

Selection views drive candidate generation and acceptance. Held-out views are used only for the final comparison. `TRANSLUCENT_MAX_OPACITY` is the safeguard: Gaussians above it are never removed by this method.

In [ ]:
import sys

for import_path in (assignment_root, assignment_root / 'scripts'):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

SH_DEGREE = 1
SELECTION_VIEWS = (0, 1, 2)
HELD_OUT_VIEWS = (3, 4)
TRANSLUCENT_MAX_OPACITY = 0.4
DARK_THRESHOLD = 0.05
EXCESS_THRESHOLD = 0.10
EVIDENCE_DILATION_PX = 4
VOXEL_SIZE_MULTIPLIER = 4.0
MIN_VIEWS_WITH_EVIDENCE = 2
MAX_CLUSTER_MEMBERS = 5000
TOP_K_CLUSTERS = 20
MAX_MAE_REGRESSION = 0.001
output_dir = assignment_root / 'report/images/ghost_region_pruning'
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Selection views: {SELECTION_VIEWS}; held-out views: {HELD_OUT_VIEWS}')
print(f'Output directory: {output_dir}')

## Load the model, cameras, references, and baseline renders

The baseline is the unmodified model rendered with the same black background as `render.py`.

In [ ]:
import PIL.Image
import numpy
import torch
import render
import gaussian_model
import ghost_region_pruning as ghost_pruning

camera_files = sorted((assignment_root / 'assets/cameras').glob('*.npy'))
cameras = []
for camera_file in camera_files:
    config = numpy.load(camera_file, allow_pickle=True).item()
    cameras.append(render.Camera(
        config['R'], config['T'], config['FoVx'], config['FoVy'],
        config['image_width'], config['image_height'],
    ))
ALL_VIEWS = tuple(range(len(cameras)))

model = gaussian_model.GaussianModel(SH_DEGREE)
model.load_ply(str(assignment_root / 'assets/gs_cloud.ply'))
gaussian_count = model.get_xyz.shape[0]
background = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32, device='cuda')

references = []
for view in ALL_VIEWS:
    reference_np = numpy.asarray(PIL.Image.open(assignment_root / f'assets/gt/{view}.png').convert('RGB'))
    reference = torch.tensor(reference_np, dtype=torch.float32, device='cuda') / 255.0
    if reference.shape[:2] != (cameras[view].image_height, cameras[view].image_width):
        raise ValueError(f'Reference {view} shape {tuple(reference.shape)} does not match its camera')
    references.append(reference)


def render_view(camera, gaussians):
    with torch.no_grad():
        image = render.render(camera, gaussians, background)['render']
    return image.permute(1, 2, 0).clamp(0.0, 1.0)


def metrics_for(image, view):
    return ghost_pruning.image_metrics(image, references[view], DARK_THRESHOLD, EXCESS_THRESHOLD)


baseline_images = {view: render_view(cameras[view], model) for view in ALL_VIEWS}
baseline_metrics = {view: metrics_for(baseline_images[view], view) for view in ALL_VIEWS}
print(f'Gaussians: {gaussian_count}')
for view in ALL_VIEWS:
    print('View {}: MAE {:.5f}, flagged pixels {}'.format(view, baseline_metrics[view]['mae'], baseline_metrics[view]['flagged_pixels']))

## Flagged pixels and translucent evidence

A pixel is flagged when the reference luminance is at most `DARK_THRESHOLD` and the baseline render is at least `EXCESS_THRESHOLD` brighter. Evidence is a Gaussian center that projects into a flagged pixel, after dilating the flagged mask by `EVIDENCE_DILATION_PX`. Only translucent Gaussians can become candidates.

In [ ]:
centers = model.get_xyz.detach()
opacity = model.get_opacity.detach().reshape(-1)
translucent = ghost_pruning.translucent_mask(opacity, TRANSLUCENT_MAX_OPACITY)
print(f'Translucent candidates (opacity <= {TRANSLUCENT_MAX_OPACITY}): {int(translucent.sum())} of {gaussian_count}')

flagged_raw = {}
evidence_rows = []
for view in SELECTION_VIEWS:
    flagged_raw[view] = ghost_pruning.flagged_pixel_mask(
        baseline_images[view], references[view], DARK_THRESHOLD, EXCESS_THRESHOLD
    )
    dilated = ghost_pruning.dilate_mask(flagged_raw[view], EVIDENCE_DILATION_PX)
    pixels, inside = ghost_pruning.project_centers(cameras[view], centers)
    evidence = ghost_pruning.evidence_mask(pixels, inside, dilated)
    evidence_rows.append(evidence)
    print('View {}: flagged pixels {}, translucent evidence Gaussians {}'.format(
        view, int(flagged_raw[view].sum()), int((evidence & translucent).sum())))

evidence_np = torch.stack(evidence_rows).cpu().numpy()

## Group evidence into voxel clusters

The voxel edge is `VOXEL_SIZE_MULTIPLIER` times the median of each Gaussian's largest scale. Only voxels that contain translucent evidence start a cluster. Touching voxels (26-connectivity) merge, and every translucent Gaussian in a merged voxel joins that cluster.

In [ ]:
median_max_scale = float(model.get_scaling.detach().max(dim=1).values.median())
voxel_size = VOXEL_SIZE_MULTIPLIER * median_max_scale
clusters = ghost_pruning.build_voxel_clusters(
    centers.cpu().numpy(), translucent.cpu().numpy(), evidence_np, voxel_size
)
selected = ghost_pruning.select_clusters(clusters, MIN_VIEWS_WITH_EVIDENCE, MAX_CLUSTER_MEMBERS, TOP_K_CLUSTERS)
print('Median largest scale: {:.5f}; voxel size: {:.5f}'.format(median_max_scale, voxel_size))
print('Clusters seeded by translucent evidence: {}; selected for ablation: {}'.format(len(clusters), len(selected)))
for cluster in selected:
    print('cluster {}: members {}, evidence Gaussians {}, views {}, hits {}'.format(
        cluster['id'], cluster['member_count'], cluster['evidence_gaussians'],
        cluster['views_with_evidence'], cluster['evidence_hits']))

## Ablate each cluster on the selection views

Each cluster is removed alone and re-rendered on the selection views. It is accepted only if the total flagged pixels drop and no selection view's MAE rises by more than `MAX_MAE_REGRESSION`.

In [ ]:
selection_before = [baseline_metrics[view] for view in SELECTION_VIEWS]
accepted = []
ablation_rows = []
for cluster in selected:
    removed = torch.as_tensor(cluster['member_indices'], dtype=torch.long, device=centers.device)
    trial = ghost_pruning.subset_model(
        model, ghost_pruning.keep_mask_without(gaussian_count, removed, centers.device)
    )
    selection_after = [metrics_for(render_view(cameras[view], trial), view) for view in SELECTION_VIEWS]
    flagged_delta = sum(after['flagged_pixels'] for after in selection_after) - sum(before['flagged_pixels'] for before in selection_before)
    mae_delta = max(after['mae'] - before['mae'] for before, after in zip(selection_before, selection_after))
    is_accepted = ghost_pruning.cluster_is_accepted(selection_before, selection_after, MAX_MAE_REGRESSION)
    if is_accepted:
        accepted.append(cluster)
    ablation_rows.append({
        'cluster_id': cluster['id'],
        'members': cluster['member_count'],
        'evidence_views': cluster['views_with_evidence'],
        'evidence_hits': cluster['evidence_hits'],
        'flagged_delta': flagged_delta,
        'max_mae_delta': mae_delta,
        'accepted': is_accepted,
    })
    print('cluster {}: flagged delta {:+d}, max MAE delta {:+.5f}, accepted {}'.format(
        cluster['id'], flagged_delta, mae_delta, is_accepted))
    del trial
    torch.cuda.empty_cache()
print('Accepted clusters: {} of {}'.format(len(accepted), len(selected)))

## Remove accepted clusters and compare on held-out views

Accepted clusters are removed together. The opacity-matched baseline removes the same number of Gaussians with the lowest opacity from all Gaussians. It controls for how many Gaussians are removed, not for where they are.

In [ ]:
if accepted:
    removed_all = torch.as_tensor(
        numpy.concatenate([cluster['member_indices'] for cluster in accepted]),
        dtype=torch.long, device=centers.device,
    )
else:
    removed_all = torch.zeros(0, dtype=torch.long, device=centers.device)
removed_count = int(removed_all.numel())
print('Removed Gaussians: {} ({:.3%} of all Gaussians)'.format(removed_count, removed_count / gaussian_count))

cluster_pruned = ghost_pruning.subset_model(
    model, ghost_pruning.keep_mask_without(gaussian_count, removed_all, centers.device)
)
opacity_matched = ghost_pruning.subset_model(
    model,
    ghost_pruning.keep_mask_without(
        gaussian_count,
        ghost_pruning.lowest_opacity_indices(opacity, removed_count),
        centers.device,
    ),
)
pruned_models = {'cluster_pruned': cluster_pruned, 'opacity_matched': opacity_matched}
images = {'baseline': baseline_images}
for name, gaussians in pruned_models.items():
    images[name] = {view: render_view(cameras[view], gaussians) for view in ALL_VIEWS}
metrics = {name: {view: metrics_for(images[name][view], view) for view in ALL_VIEWS} for name in images}

print('{:<16}{:<11}{:>5}{:>10}{:>10}{:>10}'.format('variant', 'split', 'view', 'MAE', 'PSNR', 'flagged'))
for name in images:
    for view in ALL_VIEWS:
        split = 'selection' if view in SELECTION_VIEWS else 'held-out'
        row = metrics[name][view]
        print('{:<16}{:<11}{:>5}{:>10.5f}{:>10.2f}{:>10}'.format(
            name, split, view, row['mae'], row['psnr_db'], row['flagged_pixels']))

held_out_summary = {}
for name in images:
    held = [metrics[name][view] for view in HELD_OUT_VIEWS]
    held_out_summary[name] = {
        'mae': sum(row['mae'] for row in held) / len(held),
        'psnr_db': sum(row['psnr_db'] for row in held) / len(held),
        'flagged_pixels': sum(row['flagged_pixels'] for row in held) / len(held),
    }
    print('held-out mean, {}: MAE {:.5f}, flagged {:.0f}'.format(
        name, held_out_summary[name]['mae'], held_out_summary[name]['flagged_pixels']))

## Save marked outputs

For every view, the `*_flagged.png` files mark that variant's flagged pixels in red. The baseline files also mark accepted cluster centers in cyan. `view_N_comparison.png` shows the four panels side by side. Metrics go to `metrics_by_view.csv`, `cluster_ablation.csv`, and `summary.json`.

Read the held-out rows first. The result supports the idea only if `cluster_pruned` lowers held-out flagged pixels relative to `opacity_matched` without a clear MAE regression.

In [ ]:
import csv
import json
import matplotlib.pyplot as plt


def to_uint8(image):
    return (image.clamp(0.0, 1.0) * 255.0).round().to(torch.uint8).cpu().numpy()


def overlay_centers(image_uint8, pixels_xy, color=(0, 255, 255), half_size=2):
    height, width = image_uint8.shape[:2]
    rows = numpy.clip(numpy.round(pixels_xy[:, 1]).astype(numpy.int64), 0, height - 1)
    columns = numpy.clip(numpy.round(pixels_xy[:, 0]).astype(numpy.int64), 0, width - 1)
    for offset_y in range(-half_size, half_size + 1):
        for offset_x in range(-half_size, half_size + 1):
            image_uint8[
                numpy.clip(rows + offset_y, 0, height - 1),
                numpy.clip(columns + offset_x, 0, width - 1),
            ] = color
    return image_uint8


accepted_mask = torch.zeros(gaussian_count, dtype=torch.bool, device=centers.device)
accepted_mask[removed_all] = True
for view in ALL_VIEWS:
    split = 'selection' if view in SELECTION_VIEWS else 'held-out'
    pixels, inside = ghost_pruning.project_centers(cameras[view], centers)
    accepted_pixels = pixels[inside & accepted_mask].cpu().numpy()
    flagged = {}
    marked = {}
    for name in images:
        flagged[name] = ghost_pruning.flagged_pixel_mask(images[name][view], references[view], DARK_THRESHOLD, EXCESS_THRESHOLD)
        marked[name] = ghost_pruning.mark_pixels(images[name][view], flagged[name])
    marked['baseline'] = overlay_centers(marked['baseline'], accepted_pixels)
    for name in images:
        PIL.Image.fromarray(marked[name]).save(output_dir / f'view_{view}_{name}_flagged.png')

    figure, axes = plt.subplots(1, 4, figsize=(20, 5.5))
    axes[0].imshow(to_uint8(references[view]))
    axes[0].set_title('Ground truth')
    for axis, name in zip(axes[1:], images):
        axis.imshow(marked[name])
        axis.set_title(f'{name}: {int(flagged[name].sum())} flagged px')
    for axis in axes:
        axis.axis('off')
    figure.suptitle(f'View {view} ({split})')
    figure.tight_layout()
    figure.savefig(output_dir / f'view_{view}_comparison.png', dpi=120)
    plt.show()
    plt.close(figure)

with (output_dir / 'metrics_by_view.csv').open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['variant', 'split', 'view', 'mae', 'psnr_db', 'flagged_pixels'])
    for name in images:
        for view in ALL_VIEWS:
            row = metrics[name][view]
            split = 'selection' if view in SELECTION_VIEWS else 'held_out'
            writer.writerow([name, split, view, row['mae'], row['psnr_db'], row['flagged_pixels']])

with (output_dir / 'cluster_ablation.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['cluster_id', 'members', 'evidence_views', 'evidence_hits', 'flagged_delta', 'max_mae_delta', 'accepted'])
    writer.writeheader()
    writer.writerows(ablation_rows)

summary = {
    'config': {
        'sh_degree': SH_DEGREE,
        'selection_views': list(SELECTION_VIEWS),
        'held_out_views': list(HELD_OUT_VIEWS),
        'translucent_max_opacity': TRANSLUCENT_MAX_OPACITY,
        'dark_threshold': DARK_THRESHOLD,
        'excess_threshold': EXCESS_THRESHOLD,
        'evidence_dilation_px': EVIDENCE_DILATION_PX,
        'voxel_size': voxel_size,
        'min_views_with_evidence': MIN_VIEWS_WITH_EVIDENCE,
        'max_cluster_members': MAX_CLUSTER_MEMBERS,
        'top_k_clusters': TOP_K_CLUSTERS,
        'max_mae_regression': MAX_MAE_REGRESSION,
    },
    'gaussian_count': gaussian_count,
    'removed_count': removed_count,
    'accepted_clusters': [
        {
            'id': cluster['id'],
            'members': cluster['member_count'],
            'bbox_min': cluster['bbox_min'].tolist(),
            'bbox_max': cluster['bbox_max'].tolist(),
        }
        for cluster in accepted
    ],
    'held_out_mean': held_out_summary,
}
with (output_dir / 'summary.json').open('w') as handle:
    json.dump(summary, handle, indent=2)
print('Saved outputs to {}'.format(output_dir))